# Segmentation d'instance des câbles et connecteurs (YOLOv8-seg)

Ce notebook est adapté au dataset **`wiring harness connectors segmen.v1i.yolov8`** (extrait depuis `wiring harness connectors segmen.v1i.yolov8.zip`).

### Ce que fait ce notebook
1. Décompresse automatiquement `wiring harness connectors segmen.v1i.yolov8.zip` si le dossier cible n'existe pas encore.
2. Audite et nettoie les fichiers de labels (`.txt`) pour garantir la compatibilité avec la segmentation (`yolov8n-seg.pt`), en supprimant les lignes à 4 coordonnées (*box-only*) si présentes et en régénérant le cache Ultralytics (`*.cache`).
3. Entraîne **YOLOv8-seg** (`yolov8n-seg.pt`) par **paliers de 10 époques** (jusqu'à 100 époques au total) en stockant chaque modèle intermédiaire dans `MODELS_DIR`.

## 1. Importation des bibliothèques

In [7]:
import shutil
import zipfile
from collections import Counter
from pathlib import Path
from ultralytics import YOLO

## 2. Extraction (si nécessaire) et vérification du dataset

In [8]:
DATASET_DIR = Path("../data/processed/wiring harness connectors segmen.v1i.yolov8").resolve()
ZIP_PATH = Path("../data/raw/wiring harness connectors segmen.v1i.yolov8.zip").resolve()
DATA_YAML = DATASET_DIR / "data.yaml"

if not DATASET_DIR.exists() and ZIP_PATH.exists():
    print(f"Extraction de {ZIP_PATH} vers {DATASET_DIR}...")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATASET_DIR)

if not DATA_YAML.exists():
    raise FileNotFoundError(
        f"Fichier data.yaml introuvable : {DATA_YAML}\nVérifiez le chemin de DATASET_DIR ou ZIP_PATH."
    )

print(f"Dataset prêt : {DATASET_DIR}")
print(f"Fichier trouvé : {DATA_YAML}")

Dataset prêt : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8
Fichier trouvé : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml


## 3. Audit et nettoyage des annotations pour la segmentation (`yolo_seg`)

In [9]:
CLEAN_BOX_ONLY = True

for split in ["train", "valid", "test"]:
    lbl_dir = DATASET_DIR / split / "labels"
    if not lbl_dir.exists():
        continue

    stats = Counter()
    for txt_file in lbl_dir.glob("*.txt"):
        lines = [ln.strip() for ln in txt_file.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            stats["empty_bg_files"] += 1
            continue

        valid_lines = []
        for ln in lines:
            parts = ln.split()
            n_coords = len(parts) - 1
            if n_coords >= 6 and n_coords % 2 == 0:
                stats["polygons"] += 1
                valid_lines.append(ln)
            elif n_coords == 4:
                stats["box_only"] += 1
                if not CLEAN_BOX_ONLY:
                    valid_lines.append(ln)
            else:
                stats["invalid"] += 1

        if CLEAN_BOX_ONLY and len(valid_lines) != len(lines):
            txt_file.write_text("\n".join(valid_lines) + ("\n" if valid_lines else ""), encoding="utf-8")

    for cache_file in (DATASET_DIR / split).glob("*.cache"):
        cache_file.unlink(missing_ok=True)

    print(f"[{split}] {dict(stats)}")

[train] {'polygons': 995, 'empty_bg_files': 12}
[valid] {'polygons': 106, 'empty_bg_files': 2}
[test] {'polygons': 25}


## 4. Chargement du modèle pré-entraîné (`yolov8n-seg.pt`) et préparation du stockage par paliers

In [10]:
MODELS_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models").resolve()
CHECKPOINTS_DIR = MODELS_DIR / "01_detection_et_segmentation_paliers"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_WEIGHTS = "yolov8n-seg.pt"
model = YOLO(BASE_WEIGHTS)

def train_10_epochs(weights_source, run_name: str, saved_model_name: str) -> Path:
    """Entraîne 10 époques à partir de `weights_source` et stocke le meilleur modèle du palier."""
    CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
    stage_model = YOLO(str(weights_source))
    stage_model.train(
        data=str(DATA_YAML),
        task="segment",
        epochs=10,
        imgsz=640,
        batch=8,
        patience=5,
        workers=0,
        project=str(MODELS_DIR),
        name=run_name,
        exist_ok=True,
        save=True,
        plots=True,
    )
    trainer_best = Path(stage_model.trainer.best)
    trainer_last = Path(stage_model.trainer.last)
    src_pt = trainer_best if trainer_best.exists() else trainer_last
    stored_path = CHECKPOINTS_DIR / saved_model_name
    stored_path.parent.mkdir(parents=True, exist_ok=True)
    if not src_pt.exists():
        raise FileNotFoundError(f"Fichier de poids introuvable : {src_pt}")
    shutil.copy2(src_pt, stored_path)
    print(f"Modèle stocké : {stored_path}")
    return stored_path

## 5. Entraînement par paliers de 10 époques (100 époques au total)

### Palier (1 à 10 époques)

In [ ]:
model_ep10 = train_10_epochs(
    weights_source=BASE_WEIGHTS,
    run_name="01_detection_et_segmentation/ep01_10",
    saved_model_name="best_ep10.pt",
)

model = YOLO(str(model_ep10))

In [11]:
CHECKPOINTS_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models\01_detection_et_segmentation_paliers")
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

best_ep10_src = Path(r"D:\S5\Projet_d_expertise\projet\models\01_detection_et_segmentation\ep01_10\weights\best.pt")
model_ep10 = CHECKPOINTS_DIR / "best_ep10.pt"
if best_ep10_src.exists():
    shutil.copy2(best_ep10_src, model_ep10)
model = YOLO(str(model_ep10))

### Palier (11 à 20 époques)

In [12]:
model_ep20 = train_10_epochs(
    weights_source=model_ep10,
    run_name="01_detection_et_segmentation/ep11_20",
    saved_model_name="best_ep20.pt",
)
model = YOLO(str(model_ep20))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S

### Palier (21 à 30 époques)

In [13]:
model_ep30 = train_10_epochs(
    weights_source=model_ep20,
    run_name="01_detection_et_segmentation/ep21_30",
    saved_model_name="best_ep30.pt",
)
model = YOLO(str(model_ep30))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S

### Palier (31 à 40 époques)

In [14]:
model_ep40 = train_10_epochs(
    weights_source=model_ep30,
    run_name="01_detection_et_segmentation/ep31_40",
    saved_model_name="best_ep40.pt",
)
model = YOLO(str(model_ep40))

Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S5\Projet_d_expertise\projet\models\01_detection_et_segmentation_paliers\best_ep30.pt, momentum=0.937,

### Palier (41 à 50 époques)

In [15]:
model_ep50 = train_10_epochs(
    weights_source=model_ep40,
    run_name="01_detection_et_segmentation/ep41_50",
    saved_model_name="best_ep50.pt",
)
model = YOLO(str(model_ep50))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S

### Palier (51 à 60 époques)

In [16]:
model_ep60 = train_10_epochs(
    weights_source=model_ep50,
    run_name="01_detection_et_segmentation/ep51_60",
    saved_model_name="best_ep60.pt",
)
model = YOLO(str(model_ep60))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S

### Palier (61 à 70 époques)

In [17]:
model_ep70 = train_10_epochs(
    weights_source=model_ep60,
    run_name="01_detection_et_segmentation/ep61_70",
    saved_model_name="best_ep70.pt",
)
model = YOLO(str(model_ep70))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S

### Palier (71 à 80 époques)

In [18]:
model_ep80 = train_10_epochs(
    weights_source=model_ep70,
    run_name="01_detection_et_segmentation/ep71_80",
    saved_model_name="best_ep80.pt",
)
model = YOLO(str(model_ep80))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S

### Palier (81 à 90 époques)

In [19]:
model_ep90 = train_10_epochs(
    weights_source=model_ep80,
    run_name="01_detection_et_segmentation/ep81_90",
    saved_model_name="best_ep90.pt",
)
model = YOLO(str(model_ep90))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S

### Palier (91 à 100 époques)

In [20]:
model_ep100 = train_10_epochs(
    weights_source=model_ep90,
    run_name="01_detection_et_segmentation/ep91_100",
    saved_model_name="best_ep100.pt",
)
model = YOLO(str(model_ep100))

New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\S